# Advanced NLP for finance

Machine Learning for Finance, Dr. Arun B Ayyar. This notebook produces every number on the slides.

Four parts: attention inside a small transformer, financial sentiment, named entity recognition, and document
classification and information extraction.

The sentences, pages and loan agreements are simulated by `nlp/build/make_data.py`. The sentiment data is the public
`zeroshot/twitter-financial-news-sentiment` dataset (MIT licence), downloaded by the notebook.

The notebook takes a few minutes on a GPU (Colab: Runtime, Change runtime type, T4 GPU) and much longer on a CPU.

In [1]:
import os, sys, urllib.request

if 'google.colab' in sys.modules:            # Colab ships torch, transformers and scikit-learn
    get_ipython().system('pip -q install sentence-transformers')

REPO = 'https://raw.githubusercontent.com/iitm-da/finance-ml/master/nlp/data/'
os.makedirs('data', exist_ok=True)
for f in ('ner_train.jsonl', 'ner_test.jsonl', 'pages_train.csv', 'pages_test.csv', 'loans.jsonl'):
    if not os.path.exists('data/' + f):      # not run from the repo folder: fetch the simulated files
        urllib.request.urlretrieve(REPO + f, 'data/' + f)

In [2]:
import json, re, time, warnings, logging
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, AutoModelForTokenClassification, pipeline
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
import transformers; transformers.logging.set_verbosity_error(); transformers.logging.disable_progress_bar()

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA = 'data/'
RES = {}
print('device:', DEV)

device: cuda


# 1. Attention

DistilBERT (6 layers, 12 heads a layer, 66 million weights) is small enough to open up.
The first cell shows what the model receives: word pieces, not words.

In [3]:
tok = AutoTokenizer.from_pretrained('distilbert-base-uncased')
enc = AutoModel.from_pretrained('distilbert-base-uncased', attn_implementation='eager').eval()

WORDS = ['overdraft', 'remortgage', 'chargeback', 'collateralised', 'EBITDA', 'SONIA', 'bank', 'interest']
pieces = {w: tok.tokenize(w) for w in WORDS}
for w, p in pieces.items():
    print(f'{w:16s}', p)
print('vocabulary size:', tok.vocab_size)

overdraft        ['over', '##dra', '##ft']
remortgage       ['re', '##mo', '##rt', '##ga', '##ge']
chargeback       ['charge', '##back']
collateralised   ['collateral', '##ised']
EBITDA           ['e', '##bit', '##da']
SONIA            ['sonia']
bank             ['bank']
interest         ['interest']
vocabulary size: 30522


## One head, computed by hand

Each token's vector is turned into a query and a key. The score between two tokens is the dot product of one
token's query with the other's key, divided by the square root of the head size. A softmax over the row turns
scores into weights that add to 1.

In [4]:
SENT = 'The bank raised its base rate after inflation rose.'
x = tok(SENT, return_tensors='pt')
T_ = tok.convert_ids_to_tokens(x['input_ids'][0])
with torch.no_grad():
    o = enc(**x, output_attentions=True, output_hidden_states=True)
ATT = torch.stack(o.attentions)[:, 0]          # (6 layers, 12 heads, tokens, tokens)
i_its, i_bank = T_.index('its'), T_.index('bank')

# the head in which "its" looks hardest at "bank"
flat = ATT[:, :, i_its, i_bank]
LYR, HEAD = [int(v) for v in np.unravel_index(int(flat.argmax()), flat.shape)]
print('tokens:', T_)
print(f'layer {LYR + 1}, head {HEAD + 1}: weight from "its" to "bank" = {flat.max():.3f}')

tokens: ['[CLS]', 'the', 'bank', 'raised', 'its', 'base', 'rate', 'after', 'inflation', 'rose', '.', '[SEP]']
layer 3, head 11: weight from "its" to "bank" = 0.782


In [5]:
att = enc.transformer.layer[LYR].attention
h = o.hidden_states[LYR][0]                    # what layer LYR receives: (tokens, 768)
nh, dh = 12, 768 // 12
with torch.no_grad():
    Q = att.q_lin(h).view(-1, nh, dh)[:, HEAD] # (tokens, 64) queries of this head
    K = att.k_lin(h).view(-1, nh, dh)[:, HEAD] # (tokens, 64) keys
scores = (Q @ K.T) / dh ** 0.5                 # scaled dot product
weights = torch.softmax(scores, dim=-1)
assert torch.allclose(weights, ATT[LYR, HEAD], atol=1e-4)      # same numbers the model reports
hand = pd.DataFrame({'token': T_, 'score q.k/8': scores[i_its].numpy().round(2), 'weight': weights[i_its].numpy().round(3)})
hand

,token,score q.k/8,weight
0,[CLS],3.39,0.023
1,the,4.99,0.115
2,bank,6.91,0.782
3,raised,3.72,0.032
4,its,1.45,0.003
5,base,0.78,0.002
6,rate,1.32,0.003
7,after,-0.15,0.001
8,inflation,0.46,0.001
9,rose,-1.58,0.000


## The same word in two contexts

After six layers of attention, the vector for "bank" depends on the sentence around it.

In [6]:
BANKS = [('money', 'The bank raised its base rate after inflation rose.'),
         ('money', 'She moved her savings to another bank last month.'),
         ('money', 'The bank approved the loan within two days.'),
         ('river', 'They walked along the bank of the river at dusk.'),
         ('river', 'The boat was tied to the muddy bank.'),
         ('river', 'Reeds grew thick on the far bank of the stream.')]

def word_vec(sentence, word, layer=-1):
    e = tok(sentence, return_tensors='pt')
    with torch.no_grad():
        hs = enc(**e, output_hidden_states=True).hidden_states
    return hs[layer][0, tok.convert_ids_to_tokens(e['input_ids'][0]).index(word)]

def cos_matrix(layer):
    V = torch.stack([word_vec(s, 'bank', layer) for _, s in BANKS])
    V = V / V.norm(dim=1, keepdim=True)
    return (V @ V.T).numpy()

C0, C6 = cos_matrix(0), cos_matrix(-1)
same = [(i, j) for i in range(6) for j in range(i + 1, 6) if BANKS[i][0] == BANKS[j][0]]
diff = [(i, j) for i in range(6) for j in range(i + 1, 6) if BANKS[i][0] != BANKS[j][0]]
mean = lambda C, prs: float(np.mean([C[i, j] for i, j in prs]))
print('cosine similarity between two "bank" vectors')
print(f'  input embeddings   same sense {mean(C0, same):.3f}   different sense {mean(C0, diff):.3f}')
print(f'  after 6 layers     same sense {mean(C6, same):.3f}   different sense {mean(C6, diff):.3f}')

cosine similarity between two "bank" vectors
  input embeddings   same sense 0.970   different sense 0.955
  after 6 layers     same sense 0.862   different sense 0.592


## What attention costs

Every token attends to every token, so the work grows with the square of the length.

In [7]:
encd = enc.to(DEV)
cost = {}
for n in (32, 64, 128, 256, 512):
    xb = {'input_ids': torch.randint(1000, 5000, (16, n), device=DEV), 'attention_mask': torch.ones(16, n, dtype=torch.long, device=DEV)}
    with torch.no_grad():
        for _ in range(5): encd(**xb)                  # warm up
        if DEV == 'cuda': torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(20): encd(**xb)
        if DEV == 'cuda': torch.cuda.synchronize()
    cost[n] = (time.time() - t0) / 20 * 1000
    print(f'{n:4d} tokens  {cost[n]:7.1f} ms for a batch of 16   attention cells per head {n * n:,}')
enc = enc.to('cpu')

  32 tokens     13.4 ms for a batch of 16   attention cells per head 1,024


  64 tokens     20.0 ms for a batch of 16   attention cells per head 4,096


 128 tokens     38.5 ms for a batch of 16   attention cells per head 16,384


 256 tokens     80.2 ms for a batch of 16   attention cells per head 65,536


 512 tokens    174.7 ms for a batch of 16   attention cells per head 262,144


# 2. Financial sentiment

11,931 finance tweets and headlines labelled bearish, bullish or neutral (the `zeroshot/twitter-financial-news-sentiment`
dataset, MIT licence). The published validation file is the test set here. Two thirds of the rows are neutral, so
accuracy alone flatters a model that says "neutral" every time; macro-F1 averages the three classes equally.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

HF = 'https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment/resolve/main/'
for f in ('sent_train.csv', 'sent_valid.csv'):
    if not os.path.exists(DATA + f):
        urllib.request.urlretrieve(HF + f, DATA + f)
clean = lambda s: re.sub(r'https?://\S+', '', s).strip()
tr = pd.read_csv(DATA + 'sent_train.csv'); te = pd.read_csv(DATA + 'sent_valid.csv')
tr['text'] = tr.text.map(clean); te['text'] = te.text.map(clean)
LAB = ['bearish', 'bullish', 'neutral']
ytr, yte = tr.label.values, te.label.values
print(len(tr), 'train,', len(te), 'test')
print('share of each class in test:', {LAB[k]: round(float((yte == k).mean()), 3) for k in range(3)})
tr.groupby('label').head(3).sort_values('label')

9543 train, 2388 test
share of each class in test: {'bearish': 0.145, 'bullish': 0.199, 'neutral': 0.656}


,text,label
0,$BYND - JPMorgan reels in expectations on Beyo...,0
1,$CCL $RCL - Nomura points to bookings weakness...,0
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",0
126,$ALTG: Dougherty & Company starts at Buy,1
127,$AMD - AMD's Navi shows strong adoption - BofA,1
128,$ANCUF: BMO Capital Markets ups to Outperform,1
358,$LB - MKM Partners puts a number on Victoria's...,2
359,$WING - Baird returns to Wingstop bull camp,2
360,Analysts React To FCC Decision On Intelsat C-B...,2


In [10]:
SENTI = {}
def report(name, pred):
    SENTI[name] = {'acc': float(accuracy_score(yte, pred)), 'f1': float(f1_score(yte, pred, average='macro')),
                   'f1c': f1_score(yte, pred, average=None).tolist(), 'cm': confusion_matrix(yte, pred).tolist()}
    print(f"{name:34s} accuracy {SENTI[name]['acc']:.3f}   macro-F1 {SENTI[name]['f1']:.3f}")
PRED = {}

PRED['always neutral'] = np.full(len(te), 2)
report('always neutral', PRED['always neutral'])

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr, Xte = tfidf.fit_transform(tr.text), tfidf.transform(te.text)
lr = LogisticRegression(C=10, max_iter=2000, class_weight='balanced').fit(Xtr, ytr)
PRED['TF-IDF + logistic regression'] = lr.predict(Xte)
report('TF-IDF + logistic regression', PRED['TF-IDF + logistic regression'])

always neutral                     accuracy 0.656   macro-F1 0.264


TF-IDF + logistic regression       accuracy 0.829   macro-F1 0.773


## Two models used as they come

A sentiment model trained on general tweets, and FinBERT, trained on financial news sentences.
Neither has seen a row of this training set. Their "negative" is read as bearish and "positive" as bullish.

In [11]:
def run_pipe(model_name, texts, bs=64):
    p = pipeline('text-classification', model=model_name, device=0 if DEV == 'cuda' else -1, truncation=True, max_length=128)
    out = [r['label'].lower() for r in p(list(texts), batch_size=bs)]
    del p
    return out

MAP = {'negative': 0, 'positive': 1, 'neutral': 2}
for name, mid in (('general tweet model, as is', 'cardiffnlp/twitter-roberta-base-sentiment-latest'), ('FinBERT, as is', 'ProsusAI/finbert')):
    t0 = time.time()
    PRED[name] = np.array([MAP[l] for l in run_pipe(mid, te.text)])
    report(name, PRED[name])

general tweet model, as is         accuracy 0.702   macro-F1 0.614


FinBERT, as is                     accuracy 0.716   macro-F1 0.660


## Frozen embeddings and a linear model

A sentence-embedding model turns each text into 384 numbers once. A logistic regression is then trained on those
numbers. No transformer weights change, so this trains in seconds.

In [12]:
from sentence_transformers import SentenceTransformer
emb = SentenceTransformer('BAAI/bge-small-en-v1.5', device=DEV)
t0 = time.time()
Etr = emb.encode(tr.text.tolist(), batch_size=128, normalize_embeddings=True)
Ete = emb.encode(te.text.tolist(), batch_size=128, normalize_embeddings=True)
t_embed = time.time() - t0
lr_e = LogisticRegression(C=10, max_iter=3000, class_weight='balanced').fit(Etr, ytr)
PRED['embeddings + logistic regression'] = lr_e.predict(Ete)
report('embeddings + logistic regression', PRED['embeddings + logistic regression'])
print(f'embedding {len(tr) + len(te):,} texts took {t_embed:.0f} s on {DEV}')

embeddings + logistic regression   accuracy 0.777   macro-F1 0.738
embedding 11,931 texts took 7 s on cuda


## Fine-tuning

All 66 million DistilBERT weights are updated on the labelled rows for three passes.

In [13]:
def batches(n, bs, shuffle, seed=0):
    idx = np.random.default_rng(seed).permutation(n) if shuffle else np.arange(n)
    return [idx[i:i + bs] for i in range(0, n, bs)]

def finetune(texts, labels, test_texts, base='distilbert-base-uncased', epochs=3, lr=3e-5, bs=32, seed=0, keep=False):
    torch.manual_seed(seed)
    tk = AutoTokenizer.from_pretrained(base)
    m = AutoModelForSequenceClassification.from_pretrained(base, num_labels=3).to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * int(np.ceil(len(texts) / bs))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / (0.1 * steps), max(0.0, (steps - s) / (0.9 * steps))))
    t0 = time.time()
    m.train()
    for ep in range(epochs):
        for b in batches(len(texts), bs, True, seed + ep):
            x = tk([texts[i] for i in b], padding=True, truncation=True, max_length=64, return_tensors='pt').to(DEV)
            loss = m(**x, labels=torch.tensor(labels[b]).to(DEV)).loss
            loss.backward(); opt.step(); sched.step(); opt.zero_grad()
    secs = time.time() - t0
    m.eval(); out = []
    with torch.no_grad():
        for b in batches(len(test_texts), 128, False):
            x = tk([test_texts[i] for i in b], padding=True, truncation=True, max_length=64, return_tensors='pt').to(DEV)
            out.append(m(**x).logits.softmax(-1).cpu().numpy())
    P = np.concatenate(out)
    if keep:
        return P, secs, m, tk
    del m
    return P, secs

TR, TE = tr.text.tolist(), te.text.tolist()
P_ft, t_ft = finetune(TR, ytr, TE)
PRED['DistilBERT, fine-tuned'] = P_ft.argmax(1)
report('DistilBERT, fine-tuned', PRED['DistilBERT, fine-tuned'])
print(f'fine-tuning took {t_ft:.0f} s on {DEV}')

DistilBERT, fine-tuned             accuracy 0.880   macro-F1 0.844
fine-tuning took 95 s on cuda


## How many labels?

The same three methods trained on fewer labelled rows. Each small size is repeated on three random samples.

In [14]:
SIZES = [100, 300, 1000, 3000, len(tr)]
CURVE = {'TF-IDF + logistic regression': {}, 'embeddings + logistic regression': {}, 'DistilBERT, fine-tuned': {}}
for n in SIZES:
    f = {k: [] for k in CURVE}
    for seed in range(1 if n == len(tr) else 3):
        idx = np.arange(len(tr)) if n == len(tr) else train_test_split(np.arange(len(tr)), train_size=n, stratify=ytr, random_state=seed)[0]
        v = TfidfVectorizer(ngram_range=(1, 2), min_df=1 if n < 1000 else 2, sublinear_tf=True)
        p1 = LogisticRegression(C=10, max_iter=2000, class_weight='balanced').fit(v.fit_transform(tr.text[idx]), ytr[idx]).predict(v.transform(te.text))
        p2 = LogisticRegression(C=10, max_iter=3000, class_weight='balanced').fit(Etr[idx], ytr[idx]).predict(Ete)
        if n == len(tr):
            p3 = PRED['DistilBERT, fine-tuned']
        else:
            p3 = finetune([TR[i] for i in idx], ytr[idx], TE, epochs=max(3, min(20, 3000 // n)), seed=seed)[0].argmax(1)
        for k, p in zip(CURVE, (p1, p2, p3)):
            f[k].append(float(f1_score(yte, p, average='macro')))
    for k in CURVE:
        CURVE[k][n] = float(np.mean(f[k]))
    print(f'{n:5d} labels   ' + '   '.join(f'{k.split(",")[0].split(" +")[0]:11s} {CURVE[k][n]:.3f}' for k in CURVE))

  100 labels   TF-IDF      0.389   embeddings  0.651   DistilBERT  0.510


  300 labels   TF-IDF      0.486   embeddings  0.684   DistilBERT  0.624


 1000 labels   TF-IDF      0.596   embeddings  0.693   DistilBERT  0.664


 3000 labels   TF-IDF      0.683   embeddings  0.721   DistilBERT  0.788


 9543 labels   TF-IDF      0.773   embeddings  0.738   DistilBERT  0.844


## Rows the general model gets wrong and the fine-tuned model gets right

In [15]:
g, f_ = PRED['general tweet model, as is'], PRED['DistilBERT, fine-tuned']
fix = te[(g != yte) & (f_ == yte)].assign(general=[LAB[k] for k in g[(g != yte) & (f_ == yte)]])
fix['truth'] = fix.label.map(lambda k: LAB[k])
pd.set_option('display.max_colwidth', 110)
fix[fix.text.str.len() < 90][['text', 'truth', 'general']].head(12)

,text,truth,general
0,$ALLY - Ally Financial pulls outlook,bearish,neutral
1,"$DELL $HPE - Dell, HPE targets trimmed on compute headwinds",bearish,neutral
3,$SAN: Deutsche Bank cuts to Hold,bearish,neutral
4,$SITC: Compass Point cuts to Sell,bearish,neutral
5,$TAP - Barclays cools on Molson Coors,bearish,neutral
6,$TAP: Barclays cuts to Equal Weight,bearish,neutral
9,BTIG points to breakfast pressure for Dunkin' Brands,bearish,neutral
11,Clovis Oncology downgraded to in line from outperform at Evercore ISI,bearish,neutral
13,Goldman pulls Progressive from Goldman's conviction list; shares -2.7%,bearish,neutral
14,Hanesbrands downgraded to underperform vs. neutral at BofA Merrill Lynch,bearish,neutral


# 3. Named entity recognition

3,200 simulated sentences of four kinds: payment narratives in capitals, case notes, news lines and complaint text.
Six entity types are tagged. The test sentences use sentence patterns, people, companies and places that do not
appear in the training sentences.

In [17]:
load = lambda f: [json.loads(l) for l in open(DATA + f, encoding='utf-8')]
ner_tr, ner_te = load('ner_train.jsonl'), load('ner_test.jsonl')
TYPES = ['PER', 'ORG', 'LOC', 'MONEY', 'DATE', 'ACCT']
for r in ner_te[:4]:
    print(f"[{r['genre']}] " + ' '.join(f'{t}/{g}' if g != 'O' else t for t, g in zip(r['tokens'], r['tags'])))

[narrative] FPS CREDIT FROM MATEO/B-PER CHAUDHRY/I-PER 27/B-DATE JAN/I-DATE 2024/I-DATE
[complaint] Nobody in Bristol/B-LOC could tell me why £11,207/B-MONEY was sent to Dr/B-PER Astrid/I-PER Petrov/I-PER .
[complaint] Marta/B-PER Fairweather/I-PER at the Perth/B-LOC branch promised a refund of 3,209.78/B-MONEY GBP/I-MONEY by 19/B-DATE July/I-DATE 2024/I-DATE .
[news] Leeds/B-LOC firm Glenmoor/B-ORG Capital/I-ORG &/I-ORG Sons/I-ORG named M/B-PER Kowalski/I-PER as chair on 9/B-DATE September/I-DATE 2022/I-DATE .


## Scoring

An entity counts as found only if the predicted span has the same first token, the same last token and the same type.
Half of a company name earns nothing.

In [18]:
def spans(tags):
    """BIO tags -> set of (type, first token, last token)"""
    out, start, typ = set(), None, None
    for i, t in enumerate(list(tags) + ['O']):
        if start is not None and not (t.startswith('I-') and t[2:] == typ):
            out.add((typ, start, i - 1)); start = None
        if t.startswith('B-') or (t.startswith('I-') and start is None):
            start, typ = i, t[2:]
    return out

def prf(gold, pred, types=TYPES, rows=None):
    res = {}
    rows = range(len(gold)) if rows is None else rows
    for ty in list(types) + ['all']:
        keep = (lambda s: s[0] in types) if ty == 'all' else (lambda s: s[0] == ty)
        tp = fp = fn = 0
        for i in rows:
            g, p = {s for s in gold[i] if keep(s)}, {s for s in pred[i] if keep(s)}
            tp += len(g & p); fp += len(p - g); fn += len(g - p)
        pr, rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        res[ty] = {'p': pr, 'r': rc, 'f1': 2 * pr * rc / max(pr + rc, 1e-9), 'n': tp + fn}
    return res

GOLD = [spans(r['tags']) for r in ner_te]
GENRES = ['narrative', 'note', 'news', 'complaint']
by_genre = lambda g: [i for i, r in enumerate(ner_te) if r['genre'] == g]
print('test entities:', {t: sum(s[0] == t for g in GOLD for s in g) for t in TYPES})

test entities: {'PER': 467, 'ORG': 617, 'LOC': 373, 'MONEY': 508, 'DATE': 424, 'ACCT': 142}


## Rules

Amounts, dates and account numbers have a shape, so a regular expression can describe them.

In [19]:
MON = r'(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|june?|july?|aug(?:ust)?|sep(?:tember)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)'
RULES = {
 'MONEY': r'(?:£|(?:GBP|EUR)\s)\d[\d,]*(?:\.\d+)?(?:\s?(?:m|million))?\b|\b\d[\d,]*\.\d\d\sGBP\b|\b\d+(?:\.\d+)?\smillion\spounds\b',
 'DATE': rf'\b\d{{1,2}}\s{MON}\s\d{{4}}\b|\b\d{{2}}/\d{{2}}/\d{{4}}\b|\b\d{{4}}-\d{{2}}-\d{{2}}\b|\b{MON}\s\d{{1,2}},\s\d{{4}}\b',
 'ACCT': r'\bGB\d{2}\s[A-Z]{4}(?:\s\d{4}){3}\s\d{2}\b|\b\d{2}-\d{2}-\d{2}\s\d{8}\b',
}

def char_to_tokens(tokens):
    text, offs, pos = ' '.join(tokens), [], 0
    for t in tokens:
        offs.append((pos, pos + len(t))); pos += len(t) + 1
    return text, offs

def to_span(offs, a, b, typ):
    idx = [i for i, (s, e) in enumerate(offs) if s < b and e > a]
    return (typ, idx[0], idx[-1]) if idx else None

def rule_ner(tokens):
    text, offs = char_to_tokens(tokens)
    out = set()
    for typ, pat in RULES.items():
        for m in re.finditer(pat, text, flags=re.I):
            out.add(to_span(offs, m.start(), m.end(), typ))
    return out - {None}

P_rule = [rule_ner(r['tokens']) for r in ner_te]
NER = {'rules': prf(GOLD, P_rule, ['MONEY', 'DATE', 'ACCT'])}
pd.DataFrame(NER['rules']).T.round(3)

,p,r,f1,n
MONEY,1.0,0.746,0.855,508.0
DATE,1.0,0.736,0.848,424.0
ACCT,1.0,1.000,1.000,142.0
all,1.0,0.776,0.874,1074.0


## A general NER model, as is

`dslim/bert-base-NER` was trained on news text to find people, organisations and places.

In [20]:
gen = pipeline('token-classification', model='dslim/bert-base-NER', aggregation_strategy='simple', device=0 if DEV == 'cuda' else -1)
def general_ner(tokens):
    text, offs = char_to_tokens(tokens)
    return {to_span(offs, e['start'], e['end'], e['entity_group']) for e in gen(text) if e['entity_group'] in ('PER', 'ORG', 'LOC')} - {None}
P_gen = [general_ner(r['tokens']) for r in ner_te]
NER['general'] = prf(GOLD, P_gen, ['PER', 'ORG', 'LOC'])
NER['general_genre'] = {g: prf(GOLD, P_gen, ['PER', 'ORG', 'LOC'], by_genre(g))['all'] for g in GENRES}
del gen
print(pd.DataFrame(NER['general']).T.round(3))
print('\nF1 on people, organisations and places, by kind of text')
for g in GENRES:
    print(f"  {g:10s} {NER['general_genre'][g]['f1']:.3f}")

         p      r     f1       n
PER  0.594  0.719  0.651   467.0
ORG  0.683  0.729  0.705   617.0
LOC  0.842  0.772  0.806   373.0
all  0.685  0.737  0.710  1457.0

F1 on people, organisations and places, by kind of text
  narrative  0.023
  note       0.836
  news       0.798
  complaint  0.856


## Fine-tuning a tagger

The model sees word pieces, the labels are per word. Each word's label goes on its first piece and the other
pieces are ignored in the loss.

In [21]:
TAGS = ['O'] + [f'{b}-{t}' for t in TYPES for b in 'BI']
T2I = {t: i for i, t in enumerate(TAGS)}
ntok = AutoTokenizer.from_pretrained('distilbert-base-cased')

def encode(rows):
    e = ntok([r['tokens'] for r in rows], is_split_into_words=True, padding=True, truncation=True, max_length=96, return_tensors='pt')
    lab = torch.full(e['input_ids'].shape, -100)
    for i, r in enumerate(rows):
        prev = None
        for j, w in enumerate(e.word_ids(i)):
            if w is not None and w != prev:
                lab[i, j] = T2I[r['tags'][w]]
            prev = w
    return e, lab

e1, l1 = encode(ner_te[1:2])
align = [(p, TAGS[l] if l >= 0 else 'ignored') for p, l in zip(ntok.convert_ids_to_tokens(e1['input_ids'][0]), l1[0].tolist())][1:-1]
print(ner_te[1]['tokens'])
for p, l in align:
    print(f'  {p:12s} {l}')

['Nobody', 'in', 'Bristol', 'could', 'tell', 'me', 'why', '£11,207', 'was', 'sent', 'to', 'Dr', 'Astrid', 'Petrov', '.']
  Nobody       O
  in           O
  Bristol      B-LOC
  could        O
  tell         O
  me           O
  why          O
  £1           B-MONEY
  ##1          ignored
  ,            ignored
  207          ignored
  was          O
  sent         O
  to           O
  Dr           B-PER
  As           I-PER
  ##tri        ignored
  ##d          ignored
  Pet          I-PER
  ##rov        ignored
  .            O


In [22]:
def train_tagger(rows, epochs=4, lr=5e-5, bs=32, seed=0):
    torch.manual_seed(seed)
    m = AutoModelForTokenClassification.from_pretrained('distilbert-base-cased', num_labels=len(TAGS)).to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.01)
    t0 = time.time(); m.train()
    for ep in range(epochs):
        for b in batches(len(rows), bs, True, seed + ep):
            e, lab = encode([rows[i] for i in b])
            m(**e.to(DEV), labels=lab.to(DEV)).loss.backward(); opt.step(); opt.zero_grad()
    return m.eval(), time.time() - t0

def predict_tagger(m, rows):
    out = []
    with torch.no_grad():
        for b in batches(len(rows), 64, False):
            e, lab = encode([rows[i] for i in b])
            pr = m(**e.to(DEV)).logits.argmax(-1).cpu()
            for k, i in enumerate(b):
                out.append(spans([TAGS[p] for p, l in zip(pr[k].tolist(), lab[k].tolist()) if l != -100]))
    return out

NER['curve'] = {}
for n in (100, 300, 1000, len(ner_tr)):
    m, secs = train_tagger(ner_tr[:n], epochs=4 if n >= 1000 else 10)
    P_ft_ner = predict_tagger(m, ner_te)
    NER['curve'][n] = prf(GOLD, P_ft_ner)['all']['f1']
    print(f'{n:5d} training sentences   F1 {NER["curve"][n]:.3f}   ({secs:.0f} s)')
    if n < len(ner_tr): del m
NER['ft'] = prf(GOLD, P_ft_ner)
NER['ft_genre'] = {g: prf(GOLD, P_ft_ner, TYPES, by_genre(g))['all'] for g in GENRES}
NER['t_ft'] = secs
pd.DataFrame(NER['ft']).T.round(3)

  100 training sentences   F1 0.760   (4 s)


  300 training sentences   F1 0.884   (9 s)


 1000 training sentences   F1 0.863   (14 s)


 2400 training sentences   F1 0.891   (83 s)


,p,r,f1,n
PER,0.933,0.829,0.878,467.0
ORG,0.747,0.885,0.810,617.0
LOC,0.993,0.751,0.855,373.0
MONEY,0.967,0.984,0.976,508.0
DATE,0.932,0.936,0.934,424.0
ACCT,0.959,0.986,0.972,142.0
all,0.894,0.889,0.891,2531.0


## What the fine-tuned tagger still misses

In [23]:
miss = []
for i, r in enumerate(ner_te):
    for (ty, a, b) in sorted(GOLD[i] - P_ft_ner[i]):
        got = [s for s in P_ft_ner[i] if s[1] <= b and s[2] >= a]
        miss.append({'genre': r['genre'], 'type': ty, 'entity': ' '.join(r['tokens'][a:b + 1]),
                     'predicted': '; '.join(f"{s[0]}: {' '.join(r['tokens'][s[1]:s[2] + 1])}" for s in got) or 'nothing'})
miss = pd.DataFrame(miss)
print(len(miss), 'missed entities of', sum(len(g) for g in GOLD))
print(miss.groupby(['genre', 'type']).size().sort_values(ascending=False).head(6))
miss.head(12)

281 missed entities of 2531
genre      type
news       ORG     60
narrative  PER     55
news       LOC     50
narrative  LOC     35
           DATE    19
note       PER     16
dtype: int64


,genre,type,entity,predicted
0,narrative,PER,MATEO CHAUDHRY,ORG: MATEO CHAUDHRY
1,news,LOC,Leeds,nothing
2,news,LOC,Hull,nothing
3,narrative,MONEY,"77,330.73 STERLING","DATE: STERLING; MONEY: 77,330.73"
4,news,ORG,Okafor Chaudhry Partners,ORG: Lender Okafor Chaudhry Partners
5,news,ORG,Abernethy Qureshi Group,ORG: Lender Abernethy Qureshi Group
6,news,LOC,Sheffield,nothing
7,news,LOC,Carlisle,nothing
8,narrative,PER,MEERA IYER,ORG: MEERA IYER
9,news,ORG,Ironbridge Software Partners,ORG: Lender Ironbridge Software Partners


# 4. Documents: classification

2,000 simulated pages of eight document types. A page holds one to three paragraphs of its own type and, for most
types, one or two paragraphs that several types share (governing law, notices, statement footers).
The models train on clean text. The test pages are then degraded the way a poor scan degrades text.

In [25]:
ptr, pte = pd.read_csv(DATA + 'pages_train.csv'), pd.read_csv(DATA + 'pages_test.csv')
DT = sorted(ptr.doc_type.unique())
print(len(ptr), 'train pages,', len(pte), 'test pages')
for t in ('loan_agreement', 'guarantee'):
    print(f'\n[{t}] ' + ptr[ptr.doc_type == t].text.iloc[0])

1400 train pages, 600 test pages

[loan_agreement] Interest accrues on the outstanding principal at 3.31% per annum and is payable in arrear on each repayment date. The Lender agrees to make available to the Borrower a term loan facility in an aggregate principal amount of GBP 15,800. Signed for and on behalf of the parties on 1 November 2025. If any provision is found to be unenforceable, the remaining provisions continue in full force.

[guarantee] If any provision is found to be unenforceable, the remaining provisions continue in full force. The Guarantor waives any right to require the Lender to proceed first against the principal debtor. The Guarantor irrevocably guarantees to the Lender the due payment of all sums owed by Halcyon Media Partners.


In [26]:
SWAP = {'l': '1', 'I': 'l', 'O': '0', 'o': 'c', 'e': 'c', 'S': '5', 'B': '8', 'm': 'rn', 'h': 'b', 'a': 'o', 'u': 'v', 'i': 'l', 't': 'f', 'n': 'r'}
def ocr_noise(text, p, seed=0):
    """each character is, with probability p, swapped for a look-alike, dropped, or has its space removed"""
    r = np.random.default_rng(seed)
    out = []
    for ch in text:
        if r.random() < p:
            ch = SWAP.get(ch, '' if ch == ' ' or r.random() < .5 else ch)
        out.append(ch)
    return ''.join(out)

NOISE = [0, 0.02, 0.05, 0.10, 0.20]
print(ocr_noise(pte.text[0][:150], 0.10))

Thc directcrs havc a reasonable expectation fhatthe govp has adequate resources to continue as a gcirg ccrcern. Profit before tax wos 8P 42,50,000


In [27]:
vw = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
vc = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True, min_df=2)
mw = LogisticRegression(C=10, max_iter=2000).fit(vw.fit_transform(ptr.text), ptr.doc_type)
mc = LogisticRegression(C=10, max_iter=2000).fit(vc.fit_transform(ptr.text), ptr.doc_type)
Eptr = emb.encode(ptr.text.tolist(), batch_size=64, normalize_embeddings=True)
me = LogisticRegression(C=10, max_iter=3000).fit(Eptr, ptr.doc_type)

DOC = {'word TF-IDF': {}, 'character n-gram TF-IDF': {}, 'embeddings': {}}
PRED_DOC = {}
for p in NOISE:
    noisy = [ocr_noise(t, p, seed=i) for i, t in enumerate(pte.text)]
    preds = (mw.predict(vw.transform(noisy)), mc.predict(vc.transform(noisy)),
             me.predict(emb.encode(noisy, batch_size=64, normalize_embeddings=True)))
    for k, pr in zip(DOC, preds):
        DOC[k][p] = float((pr == pte.doc_type).mean())
        PRED_DOC[(k, p)] = pr
    print(f'{p:5.0%} of characters damaged   ' + '   '.join(f'{k} {DOC[k][p]:.3f}' for k in DOC))

   0% of characters damaged   word TF-IDF 0.962   character n-gram TF-IDF 0.962   embeddings 0.953


   2% of characters damaged   word TF-IDF 0.952   character n-gram TF-IDF 0.960   embeddings 0.950


   5% of characters damaged   word TF-IDF 0.948   character n-gram TF-IDF 0.960   embeddings 0.940


  10% of characters damaged   word TF-IDF 0.948   character n-gram TF-IDF 0.962   embeddings 0.913


  20% of characters damaged   word TF-IDF 0.887   character n-gram TF-IDF 0.962   embeddings 0.737


In [28]:
cm_doc = pd.crosstab(pte.doc_type, PRED_DOC[('word TF-IDF', 0)], rownames=['truth'], colnames=['predicted']).reindex(index=DT, columns=DT, fill_value=0)
wrong = pte[PRED_DOC[('word TF-IDF', 0)] != pte.doc_type].assign(predicted=PRED_DOC[('word TF-IDF', 0)][PRED_DOC[('word TF-IDF', 0)] != pte.doc_type])
print(len(wrong), 'of', len(pte), 'clean test pages misclassified by word TF-IDF')
wrong[['doc_type', 'predicted', 'text']].head(5)

23 of 600 clean test pages misclassified by word TF-IDF


,doc_type,predicted,text
3,account_statement,card_statement,Statement period 1 March 2023 to 28 March 2023. Page 4 of 7. Please check this statement and tell us withi...
12,mortgage_offer,loan_agreement,This agreement is governed by the law of England and Wales and the parties submit to the jurisdiction of i...
30,guarantee,loan_agreement,This agreement is governed by the law of Scotland and the parties submit to the jurisdiction of its courts...
34,mortgage_offer,loan_agreement,Any notice under this document must be in writing and is deemed received two working days after posting. I...
35,loan_agreement,mortgage_offer,"If any provision is found to be unenforceable, the remaining provisions continue in full force. This agree..."


# 5. Documents: information extraction

400 simulated loan agreements. Six fields are wanted from each: borrower, lender, principal, interest rate,
final repayment date and governing law. Each agreement also holds values that look right and are wrong:
an arrangement fee, a default rate, a first repayment date, a guarantor.

In [29]:
loans = load('loans.jsonl')
FIELDS = ['borrower', 'lender', 'principal', 'rate', 'maturity', 'law']
print(loans[0]['text'], '\n')
print({f: loans[0][f] for f in FIELDS})

THIS FACILITY AGREEMENT is made on 9 June 2024. PARTIES: (1) Halcyon Commercial Finance Ltd as lender; (2) Valebrook Freight as borrower. An event of default occurs if any sum is unpaid for more than five working days. If a sum is not paid when due, default interest of 1.48% above the agreed rate applies. The Borrower shall supply its audited accounts within 180 days of each financial year end. Interest accrues on the loan at 8.31% per annum. Repayments begin on 16 July 2025 and the last repayment is due on 26 September 2026. The Lender makes available to the Borrower a term loan in a principal amount of GBP 1,996,000. The Borrower shall pay an arrangement fee of GBP 77,000 on the date of first drawdown. The Borrower shall not create any security over its assets without the consent of the Lender. Any notice must be in writing and is deemed received two working days after posting. This agreement is governed by the law of Northern Ireland. 

{'borrower': 'Valebrook Freight', 'lender': 'H

## Rules written from the usual wording

One pattern per field, written against the phrasing that most agreements use.

In [30]:
DATE_ = r'\d{1,2} [A-Z][a-z]+ \d{4}'
RX = {
 'borrower': r'made between (.+?) \(the "Borrower"\)',
 'lender': r'\(the "Borrower"\) and (.+?) \(the "Lender"\)',
 'principal': r'principal amount of (GBP [\d,]+)',
 'rate': r'accrues on the loan at (.+?) per annum',
 'maturity': rf'repaid in full on ({DATE_})',
 'law': r'governed by the law of (.+?)\.',
}
def rule_extract(text):
    out = {}
    for f, pat in RX.items():
        m = re.search(pat, text)
        out[f] = m.group(1) if m else None
    return out

norm = lambda s: re.sub(r'\s+', ' ', re.sub(r'^(the|law of) ', '', (s or '').strip().strip('."\',;:()').lower())).replace(' per annum', '').replace(' a year', '')
ok = lambda pred, gold: norm(pred) == norm(gold)
EX_rule = [rule_extract(r['text']) for r in loans]
IE = {'rules': {f: float(np.mean([ok(p[f], r[f]) for p, r in zip(EX_rule, loans)])) for f in FIELDS}}
IE['rules_none'] = {f: float(np.mean([p[f] is None for p in EX_rule])) for f in FIELDS}
print('share of agreements with the field extracted correctly')
print({f: round(v, 3) for f, v in IE['rules'].items()})

share of agreements with the field extracted correctly
{'borrower': 0.613, 'lender': 0.613, 'principal': 0.54, 'rate': 0.522, 'maturity': 0.517, 'law': 0.59}


## A question-answering model

`deepset/roberta-base-squad2` reads a passage and a question and returns the span of the passage that answers it,
with a score. The span is chosen from the model's start and end probabilities. It was trained on Wikipedia questions and has seen no loan agreement.

In [31]:
QUESTIONS = {'borrower': 'Who is the borrower?', 'lender': 'Who is the lender?',
             'principal': 'What is the principal amount of the loan?', 'rate': 'What is the rate of interest on the loan?',
             'maturity': 'On what date must the loan be repaid in full?', 'law': 'Which law governs the agreement?'}
from transformers import AutoModelForQuestionAnswering
qtok = AutoTokenizer.from_pretrained('deepset/roberta-base-squad2')
qam = AutoModelForQuestionAnswering.from_pretrained('deepset/roberta-base-squad2').to(DEV).eval()

def answer(question, context, max_len=30):
    """the span of the context with the highest start x end probability, and that probability"""
    e = qtok(question, context, return_tensors='pt', truncation='only_second', max_length=512, return_offsets_mapping=True)
    off = e.pop('offset_mapping')[0]
    in_ctx = torch.tensor([s_ == 1 for s_ in e.sequence_ids(0)])
    with torch.no_grad():
        o = qam(**e.to(DEV))
    ps = o.start_logits[0].softmax(-1).cpu() * in_ctx; pe = o.end_logits[0].softmax(-1).cpu() * in_ctx
    sc = torch.triu(ps[:, None] * pe[None, :]) - torch.triu(ps[:, None] * pe[None, :], diagonal=max_len)   # start <= end < start + 30
    i, j = divmod(int(sc.argmax()), sc.shape[1])
    return context[off[i][0]:off[j][1]], float(sc[i, j])

t0 = time.time()
EX_qa = [{f: answer(QUESTIONS[f], r['text']) for f in FIELDS} for r in loans]
t_qa = time.time() - t0
del qam
IE['qa'] = {f: float(np.mean([ok(p[f][0], r[f]) for p, r in zip(EX_qa, loans)])) for f in FIELDS}
print({f: round(v, 3) for f, v in IE['qa'].items()})
print(f'{t_qa / len(loans) * 1000:.0f} ms an agreement on {DEV}')

{'borrower': 0.853, 'lender': 1.0, 'principal': 1.0, 'rate': 0.85, 'maturity': 0.532, 'law': 0.895}
372 ms an agreement on cuda


## Where each method fails

The simulator records which of three phrasings each clause used. Phrasing 0 is the usual one.

In [32]:
VKEY = {'borrower': 'parties', 'lender': 'parties', 'principal': 'principal', 'rate': 'rate', 'maturity': 'maturity', 'law': 'law'}
byv = []
for f in FIELDS:
    for v in range(3):
        rows = [i for i, r in enumerate(loans) if r['variant'][VKEY[f]] == v]
        byv.append({'field': f, 'phrasing': v, 'agreements': len(rows),
                    'rules': np.mean([ok(EX_rule[i][f], loans[i][f]) for i in rows]),
                    'qa model': np.mean([ok(EX_qa[i][f][0], loans[i][f]) for i in rows])})
byv = pd.DataFrame(byv)
byv.round(3)

,field,phrasing,agreements,rules,qa model
0,borrower,0,245,1.0,1.000
1,borrower,1,95,0.0,1.000
2,borrower,2,60,0.0,0.017
3,lender,0,245,1.0,1.000
4,lender,1,95,0.0,1.000
5,lender,2,60,0.0,1.000
6,principal,0,216,1.0,1.000
7,principal,1,104,0.0,1.000
8,principal,2,80,0.0,1.000
9,rate,0,209,1.0,0.722


In [33]:
qa_wrong = [{'field': f, 'truth': r[f], 'qa answer': p[f][0], 'score': round(p[f][1], 2)}
            for p, r in zip(EX_qa, loans) for f in FIELDS if not ok(p[f][0], r[f])]
print(len(qa_wrong), 'wrong answers of', len(loans) * len(FIELDS))
pd.DataFrame(qa_wrong).groupby('field').head(3).sort_values('field')

348 wrong answers of 2400


,field,truth,qa answer,score
7,borrower,Larkspur Textiles Partners,Lender,0.04
12,borrower,Ironbridge Freight Partners,Kestrel Lending Limited,0.05
22,borrower,Tamworth Marine Ltd,The Borrower shall not create any security over its assets without the consent of the Lender,0.02
18,law,England and Wales,The courts of England and Wales,0.47
20,law,England and Wales,The courts of England and Wales,0.40
27,law,Northern Ireland,The courts of Northern Ireland,0.37
0,maturity,26 September 2026,16 July 2025,0.78
2,maturity,10 November 2029,27 December 2024,0.13
3,maturity,10 August 2026,3 August 2025,0.10
1,rate,SONIA plus 1.51%,1.51% per annum,0.45


## Using the score

The model's score lets the pipeline accept confident answers and send the rest to a person.

In [34]:
sc = np.array([p[f][1] for p in EX_qa for f in FIELDS]); good = np.array([ok(p[f][0], r[f]) for p, r in zip(EX_qa, loans) for f in FIELDS])
THR = [0, .1, .2, .3, .4, .5, .6, .7, .8, .9, .95]
route = [{'threshold': t, 'accepted': float((sc >= t).mean()), 'wrong among accepted': float(1 - good[sc >= t].mean())} for t in THR]
pd.DataFrame(route).round(3)

,threshold,accepted,wrong among accepted
0,0.00,1.000,0.145
1,0.10,0.962,0.120
2,0.20,0.933,0.110
3,0.30,0.892,0.097
4,0.40,0.836,0.080
5,0.50,0.755,0.058
6,0.60,0.652,0.042
7,0.70,0.516,0.027
8,0.80,0.398,0.007
9,0.90,0.171,0.000
